In [ ]:
import os

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from PIL import Image
import requests
from io import BytesIO

In [ ]:
from tqdm import tqdm

In [ ]:
data_path = "drive/MyDrive/Amazon ML 2024/Dataset"
model_path = "drive/MyDrive/Amazon ML 2024/Models"

In [ ]:
os.listdir(data_path)

In [ ]:
os.listdir(model_path)

# Data Preprocessing

In [ ]:
import re

In [ ]:
# Define the accepted units
accepted_units = {
    'width': {'centimetre', 'foot', 'inch', 'metre', 'millimetre', 'yard'},
    'depth': {'centimetre', 'foot', 'inch', 'metre', 'millimetre', 'yard'},
    'height': {'centimetre', 'foot', 'inch', 'metre', 'millimetre', 'yard'},
    'item_weight': {'gram', 'kilogram', 'microgram', 'milligram', 'ounce', 'pound', 'ton'},
    'maximum_weight_recommendation': {'gram', 'kilogram', 'microgram', 'milligram', 'ounce', 'pound', 'ton'},
    'voltage': {'kilovolt', 'millivolt', 'volt'},
    'wattage': {'kilowatt', 'watt'},
    'item_volume': {'centilitre', 'cubic foot', 'cubic inch', 'cup', 'decilitre', 'fluid ounce', 'gallon', 'imperial gallon', 'litre', 'microlitre', 'millilitre', 'pint', 'quart'}
}

In [ ]:
def process_value(value):
    if pd.isna(value):
        return "NA"

    value = str(value)
    value = value.lower()

    # Check if it's a range value
    range_match = re.match(r'\[(\d+(?:\.\d+)?),\s*(\d+(?:\.\d+)?)\]\s*(\w+)', value)
    if range_match:
        return f"{range_match.group(2)} {range_match.group(3)}"

    # Split the value into numeric part and unit
    parts = value.split()
    if len(parts) != 2:
        return "NA"

    numeric, unit = parts

    # Check if the unit is in any of the accepted unit sets
    if not any(unit in unit_set for unit_set in accepted_units.values()):
        return "NA"

    return value

def process_df_value(df):
    df['entity_value'] = df['entity_value'].apply(process_value)
    return df

In [ ]:
train = pd.read_csv(data_path + "/train.csv")
train.shape, train.columns

In [ ]:
process_df_value(train)
train.shape

In [ ]:
train.head()

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for i in range(5):
    image_link = train['image_link'].iloc[i+100]
    try:
        response = requests.get(image_link)
        img = Image.open(BytesIO(response.content))
        axes[i].imshow(img)
        axes[i].axis('off')  # Hide axes
    except Exception as e:
        print(f"Could not load image {i}: {e}")

plt.tight_layout()
plt.show()

In [ ]:
train['entity_name'].value_counts()

In [ ]:
df = pd.read_csv(data_path + "/df.csv")
df.shape, df.columns

In [ ]:
offset = 0

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for i in range(5):
    image_link = df.iloc[i+offset, 0]
    try:
        response = requests.get(image_link)
        img = Image.open(BytesIO(response.content))
        axes[i].imshow(img)
        axes[i].axis('off')  # Hide axes
    except Exception as e:
        print(f"Could not load image {i}: {e}")

plt.tight_layout()
plt.show()

print(df.iloc[offset:offset+5, 2:])
offset = offset + 5

# Data (Fine Tuning) => 1

In [ ]:
train = pd.read_csv(data_path + "/train.csv")
train.shape, train.columns

In [ ]:
df = pd.DataFrame(columns=['image_link', 'group_id', 'entity_name', 'entity_value'])
df.columns

In [ ]:
train = train[train['entity_name'] == 'item_volume']
train.shape

In [ ]:
offset = 4944

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(30, 8))
for i in range(4):
    image_link = train['image_link'].iloc[i+offset]
    try:
        response = requests.get(image_link)
        img = Image.open(BytesIO(response.content))
        axes[i].imshow(img)
        axes[i].axis('off')  # Hide axes
    except Exception as e:
        print(f"Could not load image {i}: {e}")

plt.tight_layout()
plt.show()

In [ ]:
for i in range(4):
    idx = len(df)

    df.loc[idx] = train.iloc[i+offset]
    judge = int(input(f'{train.iloc[i+offset, 2]} + {train.iloc[i+offset, 3]} => '))

    if judge == 2:
        df.iloc[idx, 3] = 'NA'
    elif judge == 3:
        df.drop(idx, inplace=True)
        df = df.reset_index(drop=True)
    elif judge == 4:
        value = input("Enter value: ")
        df.iloc[idx, 3] = value
    else:
        continue

offset += 4

In [ ]:
df.tail(5)

In [ ]:
df['entity_name'].value_counts()

In [ ]:
df.to_csv(data_path + "/df.csv", index=False)

# Data (Fine Tuning) => 2

In [ ]:
train = pd.read_csv(data_path + "/train.csv", nrows = 5000)
train.shape, train.columns

In [ ]:
df1 = pd.read_csv(data_path + "/stage1_finetuning.csv")
df1.shape, df1.columns

In [ ]:
df2 = pd.read_csv(data_path + "/stage2_finetuning.csv")
df2.shape, df2.columns

In [ ]:
train['entity_value'].isna().sum(), df1['entity_value'].isna().sum(), df2['entity_value'].isna().sum()

In [ ]:
test = pd.read_csv(data_path + "/inference.csv")
test.shape, test.columns, test['entity_value'].isna().sum()

Download Images

---



In [ ]:
image_directory = os.path.join(data_path, "test_images")
os.makedirs(image_directory, exist_ok=True)
os.listdir(data_path)

In [ ]:
# Iterate through the train dataframe and download images
for index, row in tqdm(test.iterrows(), total=len(test), desc="Downloading images"):
    image_link = row['image_link']

    try:
        response = requests.get(image_link)
        img = Image.open(BytesIO(response.content))

        # Create a unique filename
        filename = f"train_{index}.jpg"
        filepath = os.path.join(image_directory, filename)

        # Save the image
        img.save(filepath)

    except Exception as e:
        print(f"Could not download or save image from {index}: {e}")

print(f"Downloaded {len(os.listdir(image_directory))} images to {image_directory}")

In [ ]:
index = 3328
try:
    image_link = train.iloc[index, 0]
    response = requests.get(image_link)
    img = Image.open(BytesIO(response.content))

    # Create a unique filename
    filename = f"train_{index}.jpg"
    filepath = os.path.join(image_directory, filename)

    # Save the image
    img.save(filepath)

except Exception as e:
    print(f"Could not download or save image from {index}: {e}")

In [ ]:
index = 300

print(test.iloc[index, 2])
print(test.iloc[index, 3])

# Get the list of images in the directory
image_files = os.listdir(image_directory)

# Select an image file (e.g., the last one downloaded)
if image_files:
    image_file = f'train_{index}.jpg'
    image_path = os.path.join(image_directory, image_file)

    # Load the image
    try:
        img = Image.open(image_path)

        # Display the image
        plt.imshow(img)
        plt.axis('off')  # Hide axes
        plt.title(image_file)
        plt.show()
    except Exception as e:
        print(f"Could not load or display image {image_file}: {e}")
else:
    print("No images found in the directory.")

In [ ]:
def is_float(value):
    if pd.isna(value):
        return value

    parts = value.split()
    numeric, unit = parts[0], " ".join(parts[1:])

    numeric_value = float(numeric)
    if numeric_value.is_integer():
        return f'{int(numeric_value)} {unit}'
    else:
        return value

In [ ]:
print(is_float('110.0 ounce'))

In [ ]:
df1['entity_value'] = df1['entity_value'].apply(is_float)

In [ ]:
offset = 120
image_directory = os.path.join(data_path, "stage1_images")

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for i in range(5):
    image_file = f'train_{i+offset}.jpg'
    image_path = os.path.join(image_directory, image_file)
    try:
        img = Image.open(image_path)
        axes[i].imshow(img)
        axes[i].axis('off')  # Hide axes
    except Exception as e:
        print(f"Could not load or display image {image_file}: {e}")

plt.tight_layout()
plt.show()

# Phi-3.5-vision-instruct

In [ ]:
!pip install bitsandbytes

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig

In [ ]:
model_name = "microsoft/Phi-3.5-vision-instruct"

In [ ]:
# 1. Configure 4-bit Quantization (NF4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                          # Activate 4-bit loading
    bnb_4bit_quant_type="nf4",                  # Use NF4 (a superior 4-bit format)
    bnb_4bit_compute_dtype=torch.bfloat16,      # Use bfloat16 for computation
    bnb_4bit_use_double_quant=True,             # Use a second quantization for metadata
)

# 2. Load the base model with the quantization config
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    trust_remote_code=True,
    _attn_implementation="eager",
    device_map="auto"  # 'auto' will load the model onto the GPU(s)
)

In [ ]:
# Load the processor
processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)

In [ ]:
idx = 16
print(train.iloc[idx])
image_url = train['image_link'].iloc[idx]
response = requests.get(image_url)
img = Image.open(BytesIO(response.content))
plt.imshow(img)
plt.axis('off')  # Hide axes
plt.show()

In [ ]:
user_question = "What is the weight written on this product label?"
messages = [
    {"role": "user", "content": f"<|image_1|>\n{user_question}"}
]

prompt = processor.tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

In [ ]:
# Process the image and prompt together
inputs = processor(
    text=prompt,
    images=[img],
    return_tensors="pt"
).to(model.device)

print("\nInputs processed. Running inference...")

In [ ]:
# 4. RUN INFERENCE (GENERATE RESPONSE)

# Set generation arguments
generation_args = {
    "max_new_tokens": 30,
    "temperature": 0.0,
    "do_sample": False,
}

# Generate the response
generate_ids = model.generate(**inputs, **generation_args, use_cache=False)

# Decode the generated IDs, skipping special tokens
# We slice the output to only get the new tokens (the answer)
generated_text = processor.batch_decode(
    generate_ids[:, inputs['input_ids'].shape[1]:],
    skip_special_tokens=True
)[0]

print("\n--- INFERENCE COMPLETE ---")
print(f"Question: {user_question}")
print(f"Answer: {generated_text}")

In [ ]:
model.save_pretrained(os.path.join(model_path, "phi_model"), safe_serialization=False)

# SmolVLM-256M

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq

In [ ]:
model_name = "HuggingFaceTB/SmolVLM-256M-Instruct"

In [ ]:
# Load the processor
processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)

In [ ]:
# Load the model
model = AutoModelForVision2Seq.from_pretrained(
    model_name,
    trust_remote_code=True,
    _attn_implementation="eager"
)

print(model)

In [ ]:
idx = 400
print(train.iloc[idx])
image_url = train['image_link'].iloc[idx]
response = requests.get(image_url)
img = Image.open(BytesIO(response.content))
plt.imshow(img)
plt.axis('off')  # Hide axes
plt.show()

In [ ]:
# Define the prompt. Use the chat template format.
user_question = "What is the weight written on this product label?"

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"}, # Image placeholder
            {"type": "text", "text": user_question}
        ]
    }
]

# Apply the chat template to format the prompt correctly
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
print(prompt)

In [ ]:
# Process the image and prompt together
print("\nProcessing inputs...")
inputs = processor(
    text=prompt,
    images=[img],
    return_tensors="pt"
) # Ensure inputs are on the correct device and dtype


print("Inputs processed. Running inference...")

In [ ]:
# 4. RUN INFERENCE (GENERATE RESPONSE)

# Set generation arguments
generation_args = {
    "max_new_tokens": 20, # Limit output length
    "temperature": 0.0,   # Set to 0 for deterministic output
    "do_sample": False,   # Disable sampling for deterministic output
}

# Generate the response
# Use torch.inference_mode() for efficiency
with torch.inference_mode():
    generate_ids = model.generate(**inputs, **generation_args)

# 5. DECODE AND PRINT RESPONSE

# Decode only the newly generated tokens, skipping the prompt and special tokens
generated_text = processor.batch_decode(
    generate_ids[:, inputs['input_ids'].shape[1]:], # Slice to get only new tokens
    skip_special_tokens=True
)[0].strip()

print("\n--- INFERENCE COMPLETE ---")
print(f"Question: {user_question}")
print(f"Answer: {generated_text}")